In [9]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

if not os.environ.get("HF_TOKEN"):
    print("Warning: HF_TOKEN is not set. Please ensure it's in Colab secrets.")
else:
    print("HF_TOKEN successfully loaded and set.")

print("Libraries installed and credentials loaded successfully! ✓")

HF_TOKEN successfully loaded and set.
Libraries installed and credentials loaded successfully! ✓


In [10]:
import json
import hashlib
import time
from enum import Enum
from dataclasses import dataclass, field
from typing import Optional, Dict, Any, List, Tuple
from huggingface_hub import InferenceClient

class FailureType(Enum):
    HALLUCINATION    = "hallucination"
    CONTEXT_LOSS     = "context_loss"
    GOAL_DRIFT       = "goal_drift"
    EXECUTION_ERROR  = "execution_error"

@dataclass
class EvalResult:
    score: float
    failure_type: Optional[FailureType]
    explanation: str
    dimension_scores: dict = field(default_factory=dict)
    raw_response: str = ""

MISTRAL_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

EVALUATOR_SYSTEM_PROMPT = """You are an independent, objective external Auditor and Task Evaluator for autonomous robotic environments.
Your objective is to rigidly inspect an execution trace against a criteria ruleset.

CRITICAL PROTOCOLS:
1. You are NOT the agent. You are NOT planning or executing steps.
2. Evaluate what happened with zero bias or leniency.
3. You must output a valid JSON string containing exactly three keys: "score", "failure_type", and "explanation".

Allowed failure_type values:
- "hallucination"
- "context_loss"
- "goal_drift"
- "execution_error"
- null (only if score >= 0.8)
"""

META_EVOLVER_SYSTEM_PROMPT = """You are a Meta-Optimization Director. Your job is to modify an evaluation rubric's rule list to catch historical vulnerabilities earlier.
You add explicit, defensive testing checks based on a pattern of observed agent failures.
"""

In [11]:
class SEALJudge:
    def __init__(self, model_id: str = MISTRAL_MODEL):
        hf_token = os.environ.get("HF_TOKEN")
        if not hf_token:
            raise ValueError("Environment variable 'HF_TOKEN' must be set to run Mistral Inference.")
        self.client = InferenceClient(api_key=hf_token)
        self.model_id = model_id

    def _get_rubric_hash(self, rubric: Dict[str, Any]) -> str:
        """Returns short md5 string hash slice to facilitate Shreyashree's drift calculations."""
        rubric_string = json.dumps(rubric, sort_keys=True)
        return hashlib.md5(rubric_string.encode("utf-8")).hexdigest()[:8]

    def evaluate(self, trace: str, rubric: Dict[str, Any]) -> Dict[str, Any]:
        user_message = f"""RUBRIC SPECIFICATION:
{json.dumps(rubric, indent=2)}

AGENT EXECUTION TRAJECTORY TRACE:
{trace}

Generate evaluation results adhering to this exact schema layout structure:
{{
  "score": <float between 0.0 and 1.0>,
  "failure_type": <string or null>,
  "explanation": "<1-2 sentence rationale details>"
}}
"""
        max_retries = 3
        for attempt in range(max_retries):
            try:
                completion = self.client.chat.completions.create(
                    model=self.model_id,
                    messages=[
                        {"role": "system", "content": EVALUATOR_SYSTEM_PROMPT},
                        {"role": "user", "content": user_message}
                    ],
                    temperature=0.1,
                    max_tokens=300
                )
                raw_response = completion.choices[0].message.content.strip()
                return json.loads(raw_response)
            except Exception as e:
                if attempt < max_retries - 1:
                    time.sleep(5 * (attempt + 1))
                else:
                    return {
                        "score": 0.0,
                        "failure_type": "execution_error",
                        "explanation": f"Evaluation pipeline error: {str(e)}"
                    }

    def evolve_rubric(self, rubric: Dict[str, Any], failure_history: List[Dict[str, Any]]) -> Tuple[Dict[str, Any], str]:
        user_message = f"""CURRENT RUBRIC STRUCTURE:
{json.dumps(rubric, indent=2)}

HISTORICAL FAILURES OBSERVED IN ACTIVE TESTING WINDOW:
{json.dumps(failure_history, indent=2)}

INSTRUCTIONS:
Given these past failures, rewrite the rubric configuration rules array blocks to catch this failure type earlier!
Maintain the exact same JSON criteria keys, but alter or append specific 'rules' strings defensively.
Keep total cumulative weights totaling exactly 1.0.

Return the modified rubric as a clean JSON dictionary matching the original root structural schema.
"""
        try:
            completion = self.client.chat.completions.create(
                model=self.model_id,
                messages=[
                    {"role": "system", "content": META_EVOLVER_SYSTEM_PROMPT},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.2,
                max_tokens=1000
            )
            raw_response = completion.choices[0].message.content.strip()
            new_rubric = json.loads(raw_response)
            return new_rubric, self._get_rubric_hash(new_rubric)
        except Exception:
            return rubric, self._get_rubric_hash(rubric)


class JudgeFixed(SEALJudge):
    """
    Ablation Baseline Architecture. Inherits explicit evaluation pipelines,
    but bypasses the rubric optimization mutate mechanics completely.
    """
    def evolve_rubric(self, rubric: Dict[str, Any], failure_history: List[Dict[str, Any]]) -> Tuple[Dict[str, Any], str]:
        """Returns original schema configuration completely unchanged for control group baseline metrics."""
        return rubric, self._get_rubric_hash(rubric)

print("Both Judge variants compiled successfully in this notebook context ✓")

Both Judge variants compiled successfully in this notebook context ✓


In [12]:
# --- SIMULATED EXPERIMENTAL TOGGLE ---
RUN_ABLATION_CONDITION = False  # Toggle to True to test JudgeFixed automatically

if RUN_ABLATION_CONDITION:
    print(" RUNNING CONDITION: Ablation Baseline (Rigid Rubric)")
    judge = JudgeFixed()
else:
    print(" RUNNING CONDITION: SEAL Active Loop (Evolving Rubric)")
    judge = SEALJudge()

# Mock objects to mimic the execution pipeline
initial_rubric = {
    "context_retention": {"weight": 0.5, "rules": ["Check if prior step items are held"]},
    "task_validity": {"weight": 0.5, "rules": ["Verify final object resting container state"]}
}
mock_trace = "Observation: Agent picked up the mug. Step 2: Agent drops eraser into desk container instead."
mock_failures = [{"failure_type": "goal_drift", "explanation": "Swapped mug for eraser"}]

# Step 1: Run Evaluation via Mistral
eval_metrics = judge.evaluate(mock_trace, initial_rubric)
print("\n[1] Evaluation Output Matrix:")
print(json.dumps(eval_metrics, indent=2))

# Step 2: Run Rubric Evolution & Fetch the tracking Hash Code
updated_rubric, r_hash = judge.evolve_rubric(initial_rubric, mock_failures)
print(f"\n[2] Rubric ID Footprint Hash Code: {r_hash}")
print("[3] Updated Rubric JSON:")
print(json.dumps(updated_rubric, indent=2))

 RUNNING CONDITION: SEAL Active Loop (Evolving Rubric)

[1] Evaluation Output Matrix:
{
  "score": 0.0,
  "failure_type": "execution_error",
  "explanation": "Evaluation pipeline error: (Request ID: Root=1-6a38149d-3ab83600069ee0fb5e3d664f;ae232917-951d-43c5-a602-7f4740026eae)\n\nBad request:\n{'message': \"The requested model 'mistralai/Mistral-7B-Instruct-v0.3' is not a chat model.\", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}"
}

[2] Rubric ID Footprint Hash Code: a93e2e92
[3] Updated Rubric JSON:
{
  "context_retention": {
    "weight": 0.5,
    "rules": [
      "Check if prior step items are held"
    ]
  },
  "task_validity": {
    "weight": 0.5,
    "rules": [
      "Verify final object resting container state"
    ]
  }
}
